### ** Transformers para clasificación**

En este cuaderno se construye, entrena y evalúa un modelo **TransformerEncoder** para una tarea de **clasificación de texto**.

La idea general es recorrer el pipeline completo:
1. preparar texto y vocabulario,
2. representar tokens con *embeddings*,
3. incorporar **codificación posicional**,
4. procesar la secuencia con un **encoder transformer**,
5. producir una predicción de clase.

La meta no es solo ejecutar el código, sino entender **qué entra al modelo, cómo se transforma la información y por qué la arquitectura puede clasificar texto**.


In [ ]:
# Librerías requeridas para este cuaderno 
# Las que ya vienen preinstaladas están comentadas.
# Si ejecutas el cuaderno en tu entorno local, descomenta las siguientes líneas:

# !pip install -q pandas==1.3.4 numpy==1.21.4 seaborn==0.9.0 matplotlib==3.5.0 scikit-learn==0.20.1
# - Para actualizar un paquete a la última versión disponible:
# !pip install pmdarima -U
# - Para fijar un paquete en una versión concreta:
# !pip install --upgrade pmdarima==2.0.2

# Nota: si tu entorno no soporta el comando "!pip install", deja estas líneas en comentarios.


In [ ]:
# No se requiere torchtext en esta versión.
#!pip install dash-core-components==2.0.0 
#!pip install dash-table==5.0.0
#!pip install dash==2.9.3
#!pip install -Uqq dash-html-components==2.0.0
#!pip install -Uqq portalocker>=2.0.0
#!pip install -Uqq plotly


In [ ]:
# Puedes suprimir aqui los warnings generados por tu codigo
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

from tqdm import tqdm
import numpy as np
import pandas as pd
from itertools import accumulate
import matplotlib.pyplot as plt
import math
import os
import re
import csv
import urllib.request
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn

from sklearn.manifold import TSNE
from torch.utils.data import DataLoader
from torch.utils.data.dataset import random_split
from IPython.display import Markdown as md
from tqdm import tqdm
from sklearn.manifold import TSNE
import plotly.graph_objs as go
import pickle

from torch.nn.utils.rnn import pad_sequence


# Utilidades equivalentes a torchtext, pero en Python puro

def basic_english_tokenizer(text):
    """Tokenizador simple inspirado en basic_english de torchtext."""
    text = text.lower()
    # separa signos de puntuación frecuentes
    text = re.sub(r"([.,!?;:/()\"'])", r" \1 ", text)
    text = re.sub(r"[^a-z0-9.,!?;:/()\"'\\-]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()

def get_tokenizer(name="basic_english"):
    if name != "basic_english":
        raise ValueError("Solo se implementa el tokenizador 'basic_english' en esta versión.")
    return basic_english_tokenizer


class SimpleVocab:
    def __init__(self, tokens, specials=None):
        specials = specials or []
        ordered = []
        seen = set()

        for tok in specials:
            if tok not in seen:
                ordered.append(tok)
                seen.add(tok)

        for tok in tokens:
            if tok not in seen:
                ordered.append(tok)
                seen.add(tok)

        self.itos = ordered
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}
        self.default_index = 0 if self.itos else None

    def set_default_index(self, index):
        self.default_index = index

    def __getitem__(self, token):
        if token in self.stoi:
            return self.stoi[token]
        if self.default_index is None:
            raise KeyError(token)
        return self.default_index

    def __call__(self, tokens):
        return [self[token] for token in tokens]

    def __len__(self):
        return len(self.itos)

    def get_stoi(self):
        return self.stoi

    def get_itos(self):
        return self.itos


def build_vocab_from_iterator(iterator, specials=None, min_freq=1):
    counter = Counter()
    for tokens in iterator:
        counter.update(tokens)

    tokens = [tok for tok, freq in sorted(counter.items()) if freq >= min_freq]
    vocab = SimpleVocab(tokens=tokens, specials=specials or [])
    return vocab


class AGNewsDataset:
    URLS = {
        "train": "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv",
        "test": "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/test.csv",
    }

    def __init__(self, split="train", root="./data/ag_news"):
        if split not in {"train", "test"}:
            raise ValueError("split debe ser 'train' o 'test'.")

        self.split = split
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.filepath = self.root / f"{split}.csv"

        if not self.filepath.exists():
            print(f"Descargando AG_NEWS ({split})...")
            urllib.request.urlretrieve(self.URLS[split], self.filepath)

        self.data = self._load_csv(self.filepath)

    @staticmethod
    def _load_csv(path):
        rows = []
        with open(path, "r", encoding="utf-8") as f:
            reader = csv.reader(f)
            for row in reader:
                if not row:
                    continue
                label = int(row[0])
                if len(row) >= 3:
                    text = " ".join(part.strip() for part in row[1:] if part.strip())
                elif len(row) == 2:
                    text = row[1].strip()
                else:
                    text = ""
                rows.append((label, text))
        return rows

    def __iter__(self):
        return iter(self.data)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def AG_NEWS(split=None, root="./data/ag_news"):
    if split is None:
        return AGNewsDataset("train", root=root), AGNewsDataset("test", root=root)
    return AGNewsDataset(split=split, root=root)


def to_map_style_dataset(data_iter):
    if hasattr(data_iter, "__getitem__") and hasattr(data_iter, "__len__"):
        return data_iter
    return list(data_iter)


#### **1. Funciones auxiliares**

En esta sección se definen funciones de apoyo para:
- visualizar la evolución del entrenamiento,
- inspeccionar *embeddings*,
- revisar salidas intermedias del modelo.

Estas funciones no cambian la arquitectura principal, pero sí ayudan a **interpretar resultados**, detectar errores y explicar el comportamiento del modelo durante la sustentación.


In [ ]:
def plot(COST,ACC):
    
    fig, ax1 = plt.subplots()
    color = 'tab:red'
    ax1.plot(COST, color=color)
    ax1.set_xlabel('epoch', color=color)
    ax1.set_ylabel('total loss', color=color)
    ax1.tick_params(axis='y', color=color)

    ax2 = ax1.twinx()
    color = 'tab:blue'
    ax2.set_ylabel('accuracy', color=color)  
    ax2.plot(ACC, color=color)
    ax2.tick_params(axis='y', color=color)
    fig.tight_layout()  
    plt.show()

In [ ]:
def plot_embdings(my_embdings,name,vocab):
  
  fig = plt.figure()
  ax = fig.add_subplot(111, projection='3d')

  ax.scatter(my_embdings[:,0], my_embdings[:,1], my_embdings[:,2])

  for j, label in enumerate(name):
      i=vocab.get_stoi()[label]
      ax.text(my_embdings[j,0], my_embdings[j,1], my_embdings[j,2], label)

  ax.set_xlabel('X Label')
  ax.set_ylabel('Y Label')
  ax.set_zlabel('Z Label')

  plt.show()

In [ ]:
def plot_tras(words, modelo):
    tokens = tokenizer(words)

    d_model = 100

    x = torch.tensor(text_pipeline(words)).unsqueeze(0).to(device)

    x_ = modelo.emb(x) * math.sqrt(d_model)

    x = modelo.pos_encoder(x_)

    q_proj_weight = modelo.state_dict()['transformer_encoder.layers.0.self_attn.in_proj_weight'][0:embed_dim].t()
    k_proj_weight = modelo.state_dict()['transformer_encoder.layers.0.self_attn.in_proj_weight'][embed_dim:2*embed_dim].t()
    v_proj_weight = modelo.state_dict()['transformer_encoder.layers.0.self_attn.in_proj_weight'][2*embed_dim:3*embed_dim].t()

    Q = (x @ q_proj_weight).squeeze(0)
    K = (x @ k_proj_weight).squeeze(0)
    V = (x @ v_proj_weight).squeeze(0)

    scores = Q @ K.T

    row_labels = tokens
    col_labels = row_labels

    plt.figure(figsize=(10, 8))
    plt.imshow(scores.cpu().detach().numpy())
    plt.yticks(range(len(row_labels)), row_labels)
    plt.xticks(range(len(col_labels)), col_labels, rotation=90)
    plt.title("Atención producto-punto")
    plt.show()

    att = nn.Softmax(dim=1)(scores)
    plt.figure(figsize=(10, 8))
    plt.imshow(att.cpu().detach().numpy())
    plt.yticks(range(len(row_labels)), row_labels)
    plt.xticks(range(len(col_labels)), col_labels, rotation=90)
    plt.title("Atención producto-punto escalado")
    plt.show()

    head = nn.Softmax(dim=1)(scores) @ V

    tsne(x_, tokens, title="Embeddings")
    tsne(head, tokens, title="Cabeceras de atención")


def tsne(embeddings, tokens, title="Embeddings"):
    tsne = TSNE(n_components=2, random_state=0)
    tsne_result = tsne.fit_transform(embeddings.squeeze(0).cpu().detach().numpy())
    
    plt.scatter(tsne_result[:, 0], tsne_result[:, 1])

    plt.title(title)

    for j, label in enumerate(tokens):
        plt.text(tsne_result[j, 0], tsne_result[j, 1], label)

    plt.show()

In [ ]:
def save_list_to_file(lst, filename):
    with open(filename, 'wb') as file:
        pickle.dump(lst, file)

def load_list_from_file(filename):
    with open(filename, 'rb') as file:
        loaded_list = pickle.load(file)
    return loaded_list

In [ ]:
dataset = [
    (1,"Introduction to NLP"),
    (2,"Basics of PyTorch"),
    (1,"NLP Techniques for Text Classification"),
    (3,"Named Entity Recognition with PyTorch"),
    (3,"Sentiment Analysis using PyTorch"),
    (3,"Machine Translation with PyTorch"),
    (1," NLP Named Entity,Sentiment Analysis,Machine Translation "),
    (1," Machine Translation with NLP "),
    (1," Named Entity vs Sentiment Analysis  NLP "),
    (3,"he painted the car red"),
    (1,"he painted the red car")
    ]

tokenizer = get_tokenizer("basic_english")

def yield_tokens(data_iter):
    for  _,text in data_iter:
        yield tokenizer(text)

vocab = build_vocab_from_iterator(yield_tokens(dataset), specials=["<unk>"])
vocab.set_default_index(vocab["<unk>"])

In [ ]:
def text_pipeline(x):
    return vocab(tokenizer(x))

def label_pipeline(x):
    return int(x) - 1

#### **2. Cero padding**

Las secuencias de texto no suelen tener la misma longitud. Para formar *batches*, es necesario igualarlas agregando un símbolo de relleno (**padding**).

En esta parte se muestra cómo:
- convertir secuencias de distinta longitud en un tensor rectangular,
- usar el valor `0` como token de relleno,
- preparar los datos para que luego puedan entrar al modelo.

Este paso es importante porque los *DataLoader* y muchas capas de PyTorch esperan tensores con dimensiones compatibles.


In [ ]:
sequences = [torch.tensor([j for j in range(1,i)]) for i in range(2,10)]
sequences

In [ ]:
padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=0)
print(padded_sequences)

#### **3. Codificación posicional**

Un transformer procesa todos los tokens en paralelo, por lo que **no conoce el orden de la secuencia** a menos que se le entregue información posicional.

Aquí se revisa la idea de **codificación posicional**:
- primero con ejemplos sencillos de *embeddings*,
- luego sumando una señal posicional a cada token,
- y finalmente observando cómo cambia la representación al incorporar el orden.

Esta parte es clave para entender por qué el modelo distingue entre frases con las mismas palabras pero en distinto orden.


In [ ]:
mi_tokens='he painted the car red he painted the red car'

mi_index=text_pipeline(mi_tokens)
mi_index

embedding_dim=3

vocab_size=len(vocab)
print(vocab_size)

embedding = nn.Embedding(vocab_size, embedding_dim)

In [ ]:
mi_embdings=embedding(torch.tensor(mi_index)).detach().numpy()
plot_embdings(mi_embdings,tokenizer(mi_tokens),vocab)

In [ ]:
position = torch.arange(0, vocab_size, dtype=torch.float).unsqueeze(1)
position

In [ ]:
d_model=3
pe = torch.zeros(vocab_size,d_model )

In [ ]:
pe=torch.cat((position, position, position), 1)
pe

In [ ]:
samples,dim=mi_embdings.shape
samples,dim

In [ ]:
pos_embding=mi_embdings+pe[0:samples,:].numpy()

In [ ]:
plot_embdings(pos_embding,tokenizer(mi_tokens),vocab)

In [ ]:
pos_embding[3]

In [ ]:
pos_embding[-1]

In [ ]:
pe=torch.cat((0.1*position, -0.1*position, 0*position), 1)

In [ ]:
plt.plot(pe[:, 0].numpy(), label="Dimension 1")
plt.plot(pe[:, 1].numpy(), label="Dimension 2")
plt.plot(pe[:, 2].numpy(), label="Dimension 3")

plt.xlabel("Número de secuencia")
plt.legend()
plt.show()

In [ ]:
pos_embding=mi_embdings+pe[0:samples,:].numpy()
plot_embdings(pos_embding,tokenizer(mi_tokens),vocab)

In [ ]:
pe=torch.cat((torch.sin(2*3.14*position/6), 0*position+1, 0*position+1), 1)
pos_embding=mi_embdings+pe[0:samples,:].numpy()
plot_embdings(pos_embding,tokenizer(mi_tokens),vocab)

In [ ]:
pe

In [ ]:
plt.plot(pe[:, 0].numpy(), label="Dimension 1", linestyle='-')
plt.plot(pe[:, 1].numpy(), label="Dimension 2", linestyle='--')
plt.plot(pe[:, 2].numpy(), label="Dimension 3", linestyle=':')

plt.ylim([-1, 1.1])

plt.xlabel("Número de secuencia")
plt.legend()
plt.show()

In [ ]:
pe=torch.cat((torch.cos(2*3.14*position/25), torch.sin(2*3.14*position/25),  torch.sin(2*3.14*position/5)), 1)
pos_embding=mi_embdings+pe[0:samples,:].numpy()
plot_embdings(pos_embding,tokenizer(mi_tokens),vocab)

In [ ]:
plt.plot(pe[:, 0].numpy(), label="Dimension 1")
plt.plot(pe[:, 1].numpy(), label="Dimension 2")
plt.plot(pe[:, 2].numpy(), label="Dimension 3")

In [ ]:
from torch import nn

class PositionalEncoding(nn.Module):
    """
    https://pytorch.org/tutorials/beginner/transformer_tutorial.html
    """

    def __init__(self, d_model, vocab_size=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(vocab_size, d_model)
        position = torch.arange(0, vocab_size, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, : x.size(1), :]
        return self.dropout(x)

In [ ]:
mi_embdings=embedding(torch.tensor(mi_index))
mi_embdings

In [ ]:
mi_embdings.shape

In [ ]:
encoder_layer=nn.TransformerEncoderLayer(
            d_model=3,
            nhead=1,
            dim_feedforward=1,
            dropout=0,
        )

In [ ]:
out=encoder_layer(mi_embdings)
out

In [ ]:
out.mean(dim=1)

In [ ]:
params_dict = encoder_layer.state_dict()
for name, param in params_dict.items():
    print(name, param.shape)

In [ ]:
embed_dim=3
q_proj_weight = encoder_layer.state_dict()['self_attn.in_proj_weight'][0:embed_dim].t()
k_proj_weight = encoder_layer.state_dict()['self_attn.in_proj_weight'][embed_dim:2*embed_dim].t()
v_proj_weight = encoder_layer.state_dict()['self_attn.in_proj_weight'][2*embed_dim:3*embed_dim].t()

In [ ]:
Q=mi_embdings@q_proj_weight
K=mi_embdings@k_proj_weight
V=mi_embdings@v_proj_weight

In [ ]:
scores=Q@K.T/np. sqrt(embed_dim)
scores

In [ ]:
head=nn.Softmax(dim=1)(scores)@V
head

In [ ]:
transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2)

#### **4. Inspección de capas y parámetros**

En este punto se revisan los **parámetros internos** del `TransformerEncoder` para ver cómo está organizada la arquitectura.

La idea es identificar:
- pesos y sesgos de atención,
- capas lineales,
- normalización,
- y dimensiones de los tensores aprendibles.

Esta inspección ayuda a conectar la teoría del bloque transformer con su implementación real en PyTorch.


In [ ]:
params_dict = transformer_encoder.state_dict()
for name, param in params_dict.items():
    print(name, param.shape)

#### **5. Clasificación de texto**

A partir de aquí el cuaderno pasa de ejemplos conceptuales a una tarea supervisada completa.

Se construirá un clasificador que:
- recibe una secuencia de tokens,
- la representa con *embeddings* y codificación posicional,
- la procesa con un **TransformerEncoder**,
- y produce una etiqueta final para cada texto.

Conviene leer esta sección pensando en el flujo **entrada → representación → encoder → predicción**.


In [ ]:
train_iter= AG_NEWS(split="train")

In [ ]:
y,text= next(iter(train_iter ))
print(y,text)

In [ ]:
ag_news_label = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tec"}
ag_news_label[y]

In [ ]:
num_class = len(set([label for (label, text) in train_iter ]))
num_class

In [ ]:
vocab = build_vocab_from_iterator(yield_tokens(train_iter), specials=["<unk>"])
vocab.set_default_index(vocab["<unk>"])

In [ ]:
vocab(["age","hello"])

#### **6. Conjunto de datos**

En esta sección se carga el conjunto de datos y se separa en particiones para entrenamiento, validación y prueba.

El objetivo es dejar claro:
- de dónde salen los textos y sus etiquetas,
- cómo se divide el conjunto de datos,
- y por qué se necesita una partición de validación además del conjunto de prueba.

Esta organización permite entrenar el modelo y evaluar si realmente generaliza.


In [ ]:
train_iter, test_iter = AG_NEWS()

train_dataset = to_map_style_dataset(train_iter)
test_dataset = to_map_style_dataset(test_iter)

num_train = int(len(train_dataset) * 0.95)

split_train_, split_valid_ = random_split(train_dataset, [num_train, len(train_dataset) - num_train])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

#### **7. Cargador de datos**

Aquí se define el `collate_fn` y los `DataLoader` que preparan los *batches* para entrenamiento y evaluación.

Esta parte resuelve tres tareas prácticas:
- transformar texto en índices,
- aplicar *padding* a secuencias de distinta longitud,
- y agrupar ejemplos en lotes manejables por la GPU o CPU.

Es una sección importante porque conecta el dataset crudo con el modelo.


In [ ]:
from torch.nn.utils.rnn import pad_sequence

def collate_batch(batch):
    label_list, text_list = [], []
    for _label, _text in batch:
        label_list.append(label_pipeline(_label))
        text_list.append(torch.tensor(text_pipeline(_text), dtype=torch.int64))


    label_list = torch.tensor(label_list, dtype=torch.int64)
    text_list = pad_sequence(text_list, batch_first=True)


    return label_list.to(device), text_list.to(device)

In [ ]:
BATCH_SIZE = 64

train_dataloader = DataLoader(
    split_train_, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch
)
valid_dataloader = DataLoader(
    split_valid_, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch
)
test_dataloader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch
)

In [ ]:
label,seqence=next(iter(valid_dataloader ))

#### **8. Red neuronal**

En esta sección se implementa la clase `Net`, que contiene la arquitectura principal del clasificador.

Observa especialmente estos componentes:
- capa de **embeddings**,
- **codificación posicional**,
- bloque **TransformerEncoder**,
- mecanismo de agregación de la secuencia,
- capa final de clasificación.

Al leer el código, intenta identificar qué tensor entra, qué forma tiene en cada etapa y qué salida produce el modelo.


In [ ]:
class Net(nn.Module):

    def __init__(
        
        self,
        vocab_size,
        num_class,
        embedding_dim=100,
        nhead=5,
        dim_feedforward=2048,
        num_layers=6,
        dropout=0.1,
        activation="relu",
        classifier_dropout=0.1):

        super().__init__()

        self.emb = nn.Embedding(vocab_size,embedding_dim)

        self.pos_encoder = PositionalEncoding(
            d_model=embedding_dim,
            dropout=dropout,
            vocab_size=vocab_size,
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
        )
        self.classifier = nn.Linear(embedding_dim, num_class)
        self.d_model = embedding_dim

    def forward(self, x):
        x = self.emb(x) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)
        x = self.classifier(x)

        return x

In [ ]:
y,x=next(iter(train_dataloader))

In [ ]:
x

In [ ]:
emsize=64

In [ ]:
vocab_size=len(vocab)
vocab_size

In [ ]:
num_class

#### **9. Creando el modelo**

Aquí se instancia la red y se hace una primera inspección de su estructura.

Es recomendable verificar:
- tamaño del vocabulario,
- número de clases,
- dimensiones de los *embeddings*,
- número de cabeceras y capas del transformer.

Antes de entrenar, esta comprobación ayuda a detectar incompatibilidades de forma o errores de configuración.


In [ ]:
modelo = Net(vocab_size=vocab_size,num_class=4).to(device)
modelo

In [ ]:
predicted_label=modelo(x)

In [ ]:
predicted_label.shape

In [ ]:
x.shape

In [ ]:
def predict(text, text_pipeline):
    with torch.no_grad():
        text = torch.unsqueeze(torch.tensor(text_pipeline(text)),0).to(device)

        output = modelo(text)
        return ag_news_label[output.argmax(1).item() + 1]

In [ ]:
predict("I like sports and stuff",text_pipeline )

In [ ]:
def evaluate(dataloader, model_eval):
    model_eval.eval()
    total_acc, total_count= 0, 0

    with torch.no_grad():
        for idx, (label, text) in enumerate(dataloader):
            predicted_label = model_eval(text.to(device))

            total_acc += (predicted_label.argmax(1) == label).sum().item()
            total_count += label.size(0)
    return total_acc / total_count

In [ ]:
evaluate(test_dataloader, modelo)

In [ ]:
# LR=0.1

# criterion = torch.nn.CrossEntropyLoss()
# optimizer = torch.optim.SGD(modelo.parameters(), lr=LR)
# scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 1.0, gamma=0.1)

#### **10. Entrenando el modelo para 10 épocas**

En esta parte se ejecuta el ciclo de entrenamiento y se observan métricas como:
- pérdida acumulada,
- precisión por época,
- comportamiento en validación.

> Omite este paso si no tienes GPU. En ese caso, puedes cargar un modelo ya entrenado y concentrarte en el análisis de resultados.

La meta aquí no es solo obtener una buena precisión, sino entender **si el modelo está aprendiendo** y cómo evoluciona durante las épocas.


In [ ]:
# EPOCHS = 10
# cum_loss_list=[]
# acc_epoch=[]
# acc_old=0

# for epoch in tqdm(range(1, EPOCHS + 1)):
#     modelo.train()
#     cum_loss=0
#     for idx, (label, text) in enumerate(train_dataloader):
#         optimizer.zero_grad()
#         label, text=label.to(device), text.to(device)


#         predicted_label = modelo(text)
#         loss = criterion(predicted_label, label)
#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(modelo.parameters(), 0.1)
#         optimizer.step()
#         cum_loss+=loss.item()
#     print("Loss",cum_loss)

#     cum_loss_list.append(cum_loss)
#     accu_val = evaluate(valid_dataloader)
#     acc_epoch.append(accu_val)

#     if accu_val > acc_old:
#       acc_old= accu_val
#       torch.save(modelo.state_dict(), 'mi_modelo.pth')

# save_list_to_file(lst=cum_loss_list, filename="loss.pkl")
# save_list_to_file(lst=acc_epoch, filename="acc.pkl")

#### **11. Registro y análisis de resultados**

Después del entrenamiento, conviene guardar:
- el modelo entrenado,
- la pérdida acumulada por época,
- la precisión promedio por época,
- y cualquier observación sobre el comportamiento del entrenamiento.

Este registro facilita comparar experimentos, reproducir resultados y sustentar decisiones de diseño.


#### **12. Ejercicios**

Los ejercicios de esta sección buscan extender el cuaderno y comprobar que entiendes la arquitectura más allá del caso base.

La idea es pasar de un **TransformerEncoder entrenado desde cero** a variantes más avanzadas, por ejemplo usando modelos preentrenados o modificando partes del pipeline.

0. **Integrar un modelo pre-entrenado (BERT) en el código existente**

   1. Instala la librería Transformers:

      ```bash
      pip install transformers
      ```

   2. En el cuaderno, sustituye la construcción de embeddings y el `TransformerEncoder` de la clase `Net` por un backbone de BERT:

      ```python
      from transformers import BertModel, BertTokenizer

      tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
      backbone  = BertModel.from_pretrained("bert-base-uncased")
      ```

   3. Adapta la cabecera de clasificación para recibir la salida contextual de BERT y producir las etiquetas del problema.

   4. Compara el desempeño del modelo preentrenado con el transformer entrenado desde cero y comenta ventajas, costos y limitaciones.


In [ ]:
### Tus respuestas